# RF modulation classification

Classify HisarMod2019.1 signals into 26 modulations (5 families). Each example
is float32 `(2, 1024)`: I and Q. SNR (dB) is metadata, never a model input.
Run cells in order; Run All starts training. See README.md for setup.

## 1. Imports

In [ ]:
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

## 2. Modulation labels and families

Keep the dataset's raw codes. `code % 10` gives the family (e.g. 21 = 16QAM,
family 1 = QAM).

In [ ]:
# Raw label codes from the dataset readme, grouped by family (code % 10) so
# confusion matrices show within-family mistakes as blocks on the diagonal.
MODULATION_NAMES = {
    0: "BPSK",
    10: "QPSK",
    20: "8PSK",
    30: "16PSK",
    40: "32PSK",
    50: "64PSK",
    1: "4QAM",
    11: "8QAM",
    21: "16QAM",
    31: "32QAM",
    41: "64QAM",
    51: "128QAM",
    61: "256QAM",
    2: "2FSK",
    12: "4FSK",
    22: "8FSK",
    32: "16FSK",
    3: "4PAM",
    13: "8PAM",
    23: "16PAM",
    4: "AM-DSB",
    14: "AM-DSB-SC",
    24: "AM-USB",
    34: "AM-LSB",
    44: "FM",
    54: "PM",
}

# Family numbers are code % 10.
FAMILY_NAMES = {
    0: "PSK",
    1: "QAM",
    2: "FSK",
    3: "PAM",
    4: "analog",
}

## 3. Load converted data

Load the `.npy` arrays made by `scripts/convert_hisarmod.py`. Memory mapping
keeps the signals on disk until rows are read.

In [ ]:
def load_partition(data_dir, partition):
    iq = np.load(data_dir / f"{partition}_iq.npy", mmap_mode="r")
    labels = np.load(data_dir / f"{partition}_labels.npy")
    snr_db = np.load(data_dir / f"{partition}_snr_db.npy")
    return iq, labels, snr_db

## 4. Normalize each signal's power

Divide each signal by the square root of its own average power (`I**2 + Q**2`).
Per-example, so nothing leaks between examples.

In [ ]:
def normalize_power(iq):
    sample_power = np.sum(iq**2, axis=-2, keepdims=True)
    average_power = np.mean(sample_power, axis=-1, keepdims=True)
    normalized = iq / np.sqrt(average_power)
    return normalized.astype(np.float32, copy=False)

## 5. Split training/validation

Split the authors' training set 80/20, stratified by (modulation, SNR), with a
fixed seed. The test set stays untouched. Returns row indices, not copies.

In [ ]:
def split_train_validation(labels, snr_db, seed):
    train_rows, validation_rows = train_test_split(
        np.arange(len(labels)),
        test_size=0.2,
        stratify=np.column_stack((labels, snr_db)),
        random_state=seed,
    )
    return np.sort(train_rows), np.sort(validation_rows)

## 6. Measure accuracy by SNR

Average correctness at each SNR. Pass `code % 10` for family accuracy.

In [ ]:
def accuracy_by_snr(true_codes, predicted_codes, snr_db):
    correct = pd.Series(true_codes == predicted_codes, name="accuracy")
    return correct.groupby(pd.Index(snr_db, name="snr_db")).mean()

### Calculate shared metrics

Overall and per-SNR accuracy, for modulations and families.

In [ ]:
def calculate_metrics(true_codes, predicted_codes, snr_db):
    overall = {
        "modulation_accuracy": float(np.mean(true_codes == predicted_codes)),
        "family_accuracy": float(
            np.mean(true_codes % 10 == predicted_codes % 10)
        ),
    }
    by_snr = pd.DataFrame(
        {
            "modulation_accuracy": accuracy_by_snr(
                true_codes, predicted_codes, snr_db
            ),
            "family_accuracy": accuracy_by_snr(
                true_codes % 10, predicted_codes % 10, snr_db
            ),
        }
    )
    return overall, by_snr

## 7. Map raw codes to CNN class indices

Cross-entropy needs indices 0-25; convert only at the model boundary.

In [ ]:
# The CNN scores classes 0-25; the data uses raw codes (0, 10, ..., 54).
# Convert between them only here, at the model boundary.
CLASS_CODES = list(MODULATION_NAMES)
CODE_TO_INDEX = {code: index for index, code in enumerate(CLASS_CODES)}

## 8. Define the 1D CNN

Four conv blocks (conv, batch norm, ReLU) grow channels 2 → 32 → 64 → 128 → 256;
blocks 2-4 halve the length with max pooling. Global average pooling, dropout,
and a linear layer output 26 logits.

In [ ]:
class ModulationCNN(nn.Module):
    def __init__(self, n_classes=26, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            # (batch, 2, 1024) -> (batch, 32, 1024)
            nn.Conv1d(2, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            # -> (batch, 64, 512)
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 128, 256)
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 256, 128)
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 256)
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, iq):
        features = self.features(iq)
        logits = self.classifier(features)
        return logits

## 9. Train the CNN

Set training options here, e.g. `TrainingParameters(epochs=10, batch_size=64)`.
Tune on validation data only.

In [ ]:
@dataclass
class TrainingParameters:
    data_dir: Path = Path("data/processed")
    seed: int = 42
    batch_size: int = 128
    learning_rate: float = 0.0003
    dropout: float = 0.3
    epochs: int = 100
    patience: int = 12
    scheduler_factor: float = 0.5
    scheduler_patience: int = 4
    min_learning_rate: float = 1e-6


training_params = TrainingParameters()

### Dataset

Reads, normalizes, and labels one signal at a time from the memory-mapped array.

In [ ]:
class SignalDataset(Dataset):
    def __init__(self, iq, labels, rows):
        self.iq = iq
        self.labels = labels
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        signal = torch.from_numpy(normalize_power(self.iq[row]))
        return signal, CODE_TO_INDEX[int(self.labels[row])]

### Train for one epoch

Update weights on each batch; return the mean training loss.

In [ ]:
def run_training_epoch(model, device, train_loader, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for signals, targets in train_loader:
        signals, targets = signals.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(signals)
        loss = nn.functional.cross_entropy(logits, targets)
        torch.autograd.backward(loss)
        optimizer.step()
        total_loss += loss.item() * len(targets)
        total_examples += len(targets)
    return total_loss / total_examples

### Evaluate

Mean loss and accuracy without updating weights (no dropout, no gradients).

In [ ]:
def evaluate(model, device, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for signals, targets in data_loader:
            signals, targets = signals.to(device), targets.to(device)
            logits = model(signals)
            total_loss += nn.functional.cross_entropy(
                logits, targets, reduction="sum"
            ).item()
            total_correct += int((logits.argmax(dim=1) == targets).sum().item())
            total_examples += len(targets)
    return total_loss / total_examples, total_correct / total_examples

### Training loop

Run epochs, halve the learning rate when validation loss plateaus, save the
best weights, and stop early after `patience` epochs without improvement.

In [ ]:
def train(
    training_params, device, train_loader, validation_loader, checkpoint_path
):
    if training_params.epochs < 1 or training_params.patience < 1:
        raise ValueError("epochs and patience must be at least one")
    model = ModulationCNN(dropout=training_params.dropout).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=training_params.learning_rate
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=training_params.scheduler_factor,
        patience=training_params.scheduler_patience,
        min_lr=training_params.min_learning_rate,
    )
    history = []
    best_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, training_params.epochs + 1):
        training_loss = run_training_epoch(
            model, device, train_loader, optimizer
        )
        validation_loss, validation_accuracy = evaluate(
            model, device, validation_loader
        )
        scheduler.step(validation_loss)
        history.append(
            {
                "epoch": epoch,
                "train_loss": training_loss,
                "validation_loss": validation_loss,
                "validation_accuracy": validation_accuracy,
            }
        )
        print(
            f"Epoch {epoch}: train {training_loss:.4f}, "
            f"val {validation_loss:.4f}, accuracy {validation_accuracy:.3f}"
        )
        if validation_loss < best_loss:
            best_loss = validation_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= training_params.patience:
            print("Early stopping.")
            break

    model.load_state_dict(
        torch.load(checkpoint_path, map_location=device, weights_only=True)
    )
    model.eval()
    return model, history

### Seeds, device, and data loaders

Seed everything; use CUDA if available, otherwise CPU.

In [ ]:
random.seed(training_params.seed)
np.random.seed(training_params.seed)
torch.manual_seed(training_params.seed)
torch.cuda.manual_seed_all(training_params.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
iq, labels, snr_db = load_partition(training_params.data_dir, "train")
train_rows, validation_rows = split_train_validation(
    labels, snr_db, training_params.seed
)
train_loader = DataLoader(
    SignalDataset(iq, labels, train_rows),
    batch_size=training_params.batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(training_params.seed),
    num_workers=0,
)
validation_loader = DataLoader(
    SignalDataset(iq, labels, validation_rows),
    batch_size=training_params.batch_size,
    shuffle=False,
    num_workers=0,
)
print(
    f"Device: {device}; train: {len(train_rows)}; val: {len(validation_rows)}"
)

### Output paths

Settings, history, and figures go to `runs/`; best weights to `models/best.pt`.
Rerunning overwrites them.

In [ ]:
run_dir = Path("runs")
model_dir = Path("models")
figures_dir = run_dir / "figures"
figures_dir.mkdir(exist_ok=True)
checkpoint_path = model_dir / "best.pt"
config = {
    **asdict(training_params),
    "data_dir": str(training_params.data_dir),
    "device": str(device),
    "class_codes": CLASS_CODES,
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")

### Run training

**This cell (and Run All) starts training.**

In [ ]:
model, history = train(
    training_params, device, train_loader, validation_loader, checkpoint_path
)

### Save the history

In [ ]:
(run_dir / "metrics.json").write_text(json.dumps(history, indent=2) + "\n")
print(f"Best weights: {checkpoint_path}; metrics: {run_dir / 'metrics.json'}")

### Training and validation loss

Rising validation loss with falling training loss suggests overfitting.

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Training")
plt.plot(history_df["epoch"], history_df["validation_loss"], label="Validation")

plt.title("Training and validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    figures_dir / "training_validation_loss.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation predictions

Predict every validation example with the best model and convert back to raw
codes. Arrays stay aligned with `validation_snr_db`.

In [ ]:
model.eval()
prediction_batches = []

with torch.no_grad():
    for signals, _ in validation_loader:
        signals = signals.to(device)
        logits = model(signals)
        predicted_indices = logits.argmax(dim=1)
        prediction_batches.append(predicted_indices.cpu().numpy())

predicted_indices = np.concatenate(prediction_batches)

# Convert CNN class indices back to the dataset's raw modulation codes.
class_codes = np.array(CLASS_CODES, dtype=np.int64)
predicted_codes = class_codes[predicted_indices]

# These follow the same row order as the validation loader.
true_codes = labels[validation_rows]
validation_snr_db = snr_db[validation_rows]

np.savez(
    run_dir / "validation_predictions.npz",
    true_codes=true_codes,
    predicted_codes=predicted_codes,
    snr_db=validation_snr_db,
)

### Validation metrics

In [ ]:
overall_metrics, metrics_by_snr = calculate_metrics(
    true_codes, predicted_codes, validation_snr_db
)
(run_dir / "validation_metrics.json").write_text(
    json.dumps(overall_metrics, indent=2) + "\n"
)
metrics_by_snr.to_csv(run_dir / "validation_accuracy_by_snr.csv")

print(pd.Series(overall_metrics).to_string())
metrics_by_snr

### Modulation accuracy versus SNR

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    metrics_by_snr.index,
    metrics_by_snr["modulation_accuracy"],
    marker="o",
    label="CNN",
)
ax.set_title("Modulation accuracy")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig(
    figures_dir / "modulation_accuracy_by_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Family accuracy versus SNR

A wrong modulation in the right family still counts as correct here.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    metrics_by_snr.index,
    metrics_by_snr["family_accuracy"],
    marker="o",
    label="CNN",
)
ax.set_title("Family accuracy")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig(
    figures_dir / "family_accuracy_by_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Confusion matrices at low and high SNR

Rows are true classes, columns predictions, normalized per row (0.80 = 80%).
Split at 0 dB.

In [ ]:
low_snr = validation_snr_db < 0
high_snr = validation_snr_db >= 0

### Validation families — SNR < 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[low_snr] % 10,
    predicted_codes[low_snr] % 10,
    labels=list(FAMILY_NAMES),
    normalize="true",
)

plt.figure(figsize=(6, 6))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(5), list(FAMILY_NAMES.values()), rotation=90)
plt.yticks(range(5), list(FAMILY_NAMES.values()))
plt.xlabel("Predicted family")
plt.ylabel("True family")
plt.title("Validation families — SNR < 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "family_confusion_low_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation families — SNR ≥ 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[high_snr] % 10,
    predicted_codes[high_snr] % 10,
    labels=list(FAMILY_NAMES),
    normalize="true",
)

plt.figure(figsize=(6, 6))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(5), list(FAMILY_NAMES.values()), rotation=90)
plt.yticks(range(5), list(FAMILY_NAMES.values()))
plt.xlabel("Predicted family")
plt.ylabel("True family")
plt.title("Validation families — SNR ≥ 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "family_confusion_high_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation modulations — SNR < 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[low_snr],
    predicted_codes[low_snr],
    labels=list(MODULATION_NAMES),
    normalize="true",
)

plt.figure(figsize=(12, 12))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(26), list(MODULATION_NAMES.values()), rotation=90)
plt.yticks(range(26), list(MODULATION_NAMES.values()))
plt.xlabel("Predicted modulation")
plt.ylabel("True modulation")
plt.title("Validation modulations — SNR < 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "modulation_confusion_low_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation modulations — SNR ≥ 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[high_snr],
    predicted_codes[high_snr],
    labels=list(MODULATION_NAMES),
    normalize="true",
)

plt.figure(figsize=(12, 12))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(26), list(MODULATION_NAMES.values()), rotation=90)
plt.yticks(range(26), list(MODULATION_NAMES.values()))
plt.xlabel("Predicted modulation")
plt.ylabel("True modulation")
plt.title("Validation modulations — SNR ≥ 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "modulation_confusion_high_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

## 10. Feature baseline — planned

Cumulants and amplitude/phase statistics with a HistGradientBoostingClassifier,
using the same split and metrics as the CNN.

## 11. Compare and evaluate once — planned

Compare both models on validation, freeze them, then evaluate once on the test
set (opt-in only).